In [35]:
import os
from dotenv import load_dotenv, dotenv_values
from groq import Groq

import os
load_dotenv(override=True)



client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [36]:
def llm(prompt:str) -> str:
    chat_completion = client.chat.completions.create(
        messages= [{
            'role': 'system', 'content': 'always talk in british accent',
            'role': 'user', 'content': prompt
            }],
        model="llama-3.3-70b-versatile",
    )

    return chat_completion.choices[0].message.content

In [37]:
question = "heyyy mate!! I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Heyy mate. I'd be happy to help you with your question. However, I need a bit more information about the course you're interested in joining. Could you please provide more details, such as the course name, institution, or any other relevant information? That way, I can assist you better and guide you through the process of joining the course.


In [38]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [39]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [40]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [41]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [42]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [43]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [44]:
len(documents)

1208

In [45]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [46]:
print(question)
index.search(question)

heyyy mate!! I just discovered the course. Can I join now?


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
 

In [47]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [48]:
def search(question):
    
    boost_dict={'question': 2.0, 'section': 0.5}
    filter_dict={'course': 'llm-zoomcamp'}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [49]:
search_results = search(question)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [50]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context. and always talk in british english accent

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [51]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [52]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [53]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [54]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
heyyy mate!! I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting

In [55]:
response = client.chat.completions.create(
        messages= [{
            'role': 'system', 'content': 'always talk in british accent',
            'role': 'user', 'content': prompt
            }],
        model="llama-3.3-70b-versatile",
)

In [56]:
# print(response.choices[0].message.content)
response.usage

CompletionUsage(completion_tokens=73, prompt_tokens=370, total_tokens=443, completion_time=0.232617392, completion_tokens_details=None, prompt_time=0.019887154, prompt_tokens_details=None, queue_time=0.055542462, total_time=0.252504546)

In [57]:
def llm(instructions, user_prompt):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        messages= message_history,
        model="llama-3.3-70b-versatile",
    )

    return response.choices[0].message.content

In [58]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt)

    return answer

In [59]:
answer = rag(question)
print(answer)

Yes, you can join the course now. However, to receive a certificate, you need to submit your project while submissions are still being accepted. Keep in mind that you'll also need to be part of a "live" cohort, as certificates are not awarded for self-paced modes. Additionally, you'll need to peer-review 3 capstone projects after submitting your own. Homework is not mandatory but is recommended for reinforcing concepts.
